# LazyRegressor screening — combined dataset
Screens 24 curated `sklearn`/`lightgbm` regressors on `delay_s` for the `combined` dataset
(Gold + Green + Red; Clough is excluded — its stop ID can't be recovered causally), tunes the
top 3 with Optuna, and saves each tuned model's test predictions. Follows
`plan/04_models/01_protocol.md` + `02_lazy.md` (L1-L7, v2).

Split v2: whole service days held out as test (2026-03-04, 2026-03-09); `cv_group` (tuning
groups inside train) is the calendar date. Dataset size, feature count and group counts are
printed below (L1) rather than hard-coded here — they change whenever the dataset is rebuilt.

The test split is touched exactly once per final model: predict, then `save_result`. Every
screening and tuning step below uses only `d.X_train` / `d.y_train` (or group-disjoint
subsamples of it).

## L1 — Setup
Imports, `set_seed(42)`, load the model-ready `combined` data.

In [1]:
import sys
import time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from lazypredict.Supervised import LazyRegressor

from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, Lars, LassoLars,
    OrthogonalMatchingPursuit, BayesianRidge, HuberRegressor, SGDRegressor,
    PassiveAggressiveRegressor,
)
from sklearn.svm import LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor,
    GradientBoostingRegressor, HistGradientBoostingRegressor, AdaBoostRegressor,
)
from sklearn.neural_network import MLPRegressor
from sklearn.dummy import DummyRegressor
from lightgbm import LGBMRegressor

from mc_common import (
    SMOKE, THREADS, budget, set_seed, load_model_ready, group_val_split, metrics, save_result,
)

set_seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)

DS = "combined"
d = load_model_ready(DS)

print("SMOKE:", SMOKE, " THREADS:", THREADS)
print("train:", len(d.X_train), " test (untouched until L6):", len(d.X_test), " features:", len(d.features))
print("train cv groups (dates):", len(np.unique(d.g_train)), sorted(np.unique(d.g_train).tolist()))

SMOKE: False  THREADS: 2
train: 81258  test (untouched until L6): 29141  features: 50
train cv groups (dates): 5 ['2026-03-03', '2026-03-05', '2026-03-06', '2026-03-10', '2026-03-16']


## L2 — Inner split for screening
Group-disjoint split of TRAIN (`group_val_split`), then subsampled further for the
LazyRegressor screening pass so 24 models fit quickly: inner-train ≤ `budget(15000, 2000)`
rows, inner-val ≤ `budget(6000, 1000)` rows. The real test split is not touched here.

In [2]:
def subsample(X, y, g, n, seed=42):
    '''Positional random subsample of (X, y[, g]) to at most n rows -- same rows in all of them.'''
    if len(X) <= n:
        return X, y, g
    rng = np.random.RandomState(seed)
    pos = np.sort(rng.choice(len(X), size=n, replace=False))
    return X.iloc[pos], y.iloc[pos], (g[pos] if g is not None else None)


X_tr, X_val, y_tr, y_val, g_tr = group_val_split(d.X_train, d.y_train, d.g_train, frac=0.2, seed=42)

X_screen_tr, y_screen_tr, _ = subsample(X_tr, y_tr, None, budget(15000, 2000), seed=42)
X_screen_val, y_screen_val, _ = subsample(X_val, y_val, None, budget(6000, 1000), seed=42)

print("inner-train:", len(X_tr), "-> screening subsample:", len(X_screen_tr))
print("inner-val:  ", len(X_val), "-> screening subsample:", len(X_screen_val))

inner-train: 50096 -> screening subsample: 15000
inner-val:   31162 -> screening subsample: 6000


## L3 — Screening
`LazyRegressor` over the curated regressor list (it fits each inside its own
`StandardScaler`+model pipeline internally, so no manual scaling is needed here).

**Excluded + why:** `XGBRegressor` (has its own notebook, `xgboost_combined.ipynb`);
`SVR`/`NuSVR`/`KernelRidge`/`GaussianProcess*` (O(n²)-O(n³) training cost, not viable at this
train-set size); `QuantileRegressor`/`TheilSenRegressor`/`RANSACRegressor` (slow or
unstable at this scale); `PoissonRegressor`/`GammaRegressor`/`TweedieRegressor` (need a
strictly positive target — `delay_s` can be negative when a bus runs early).

In [3]:
CURATED = [
    LinearRegression, Ridge, Lasso, ElasticNet, Lars, LassoLars, OrthogonalMatchingPursuit,
    BayesianRidge, HuberRegressor, SGDRegressor, PassiveAggressiveRegressor, LinearSVR,
    KNeighborsRegressor, DecisionTreeRegressor, ExtraTreeRegressor, RandomForestRegressor,
    ExtraTreesRegressor, BaggingRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, AdaBoostRegressor, MLPRegressor, LGBMRegressor,
    DummyRegressor,
]
CURATED_MAP = {cls.__name__: cls for cls in CURATED}

lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=42, regressors=CURATED, n_jobs=THREADS)
leaderboard, _ = lazy.fit(X_screen_tr, X_screen_val, y_screen_tr, y_screen_val)

if lazy.errors:
    print("models that failed to fit and were skipped:", list(lazy.errors))

leaderboard

,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
DummyRegressor,-9.651777e-03,-1.236609e-03,3.375843e+02,0.066466
OrthogonalMatchingPursuit,-2.067347e-02,-1.216644e-02,3.394219e+02,0.073578
ElasticNet,-7.812634e-02,-6.914046e-02,3.488440e+02,0.136834
GradientBoostingRegressor,-9.951964e-02,-9.035545e-02,3.522880e+02,7.033391
MLPRegressor,-9.952131e-02,-9.035710e-02,3.522883e+02,13.374604
LassoLars,-1.183017e-01,-1.089810e-01,3.552842e+02,0.083094
HistGradientBoostingRegressor,-1.444762e-01,-1.349373e-01,3.594180e+02,3.033217
ExtraTreesRegressor,-1.516385e-01,-1.420399e-01,3.605408e+02,20.487137
LGBMRegressor,-1.709449e-01,-1.611854e-01,3.635504e+02,11.300191


## L4 — Pick top 3
Rank by validation R² (descending), excluding `DummyRegressor` (the naive baseline, kept in
the leaderboard for reference but never a real candidate).

In [4]:
ranked = leaderboard.drop(index="DummyRegressor", errors="ignore").sort_values("R-Squared", ascending=False)
top3 = ranked.head(3).index.tolist()
print("top 3 by validation R2:", top3)
ranked.head(10)

top 3 by validation R2: ['OrthogonalMatchingPursuit', 'ElasticNet', 'GradientBoostingRegressor']


,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
OrthogonalMatchingPursuit,-0.020673,-0.012166,339.421862,0.073578
ElasticNet,-0.078126,-0.069140,348.843975,0.136834
GradientBoostingRegressor,-0.099520,-0.090355,352.288035,7.033391
MLPRegressor,-0.099521,-0.090357,352.288302,13.374604
LassoLars,-0.118302,-0.108981,355.284194,0.083094
HistGradientBoostingRegressor,-0.144476,-0.134937,359.417963,3.033217
ExtraTreesRegressor,-0.151638,-0.142040,360.540849,20.487137
LGBMRegressor,-0.170945,-0.161185,363.550398,11.300191
RandomForestRegressor,-0.193527,-0.183579,367.039284,23.825929


## L5 — Tune each of the top 3
Optuna per model: `n_trials=budget(40, 3)`, `timeout=budget(900, 30)`s (v2: was 600), on a
train subsample ≤ `budget(30000, 3000)` rows, scored with `GroupKFold(3)` on `d.g_train`
groups (objective = mean MAE across folds). Linear/KNN/MLP/SVR models run inside
`Pipeline([StandardScaler(), model])`; tree/ensemble models don't need scaling. Search spaces
cover every CURATED model (excluding `DummyRegressor`) so tuning works regardless of which 3
models screening picked.

`HistGradientBoostingRegressor` fixes `early_stopping=False` (its default holds out a random,
non-grouped 10% of whatever it's given — including the grouped CV fold here — which leaks
across `cv_group`) and tunes `max_iter` directly instead.

In [5]:
SCALED_MODELS = {
    "LinearRegression", "Ridge", "Lasso", "ElasticNet", "Lars", "LassoLars",
    "OrthogonalMatchingPursuit", "BayesianRidge", "HuberRegressor", "SGDRegressor",
    "PassiveAggressiveRegressor", "LinearSVR", "KNeighborsRegressor", "MLPRegressor",
}

# fixed (non-tuned) kwargs per model: convergence/verbosity knobs the search doesn't need to touch
FIXED_EXTRA = {
    "MLPRegressor": {"early_stopping": True, "max_iter": 300},
    "LGBMRegressor": {"verbose": -1, "subsample_freq": 1},
    "SGDRegressor": {"max_iter": 2000, "tol": 1e-3},
    "PassiveAggressiveRegressor": {"max_iter": 2000, "tol": 1e-3},
    "LinearSVR": {"max_iter": 5000},
    "HuberRegressor": {"max_iter": 500},
}


def _fixed_kwargs(cls):
    '''random_state/n_jobs, only for the estimators whose constructor accepts them.'''
    import inspect
    p = inspect.signature(cls.__init__).parameters
    kw = {}
    if "random_state" in p:
        kw["random_state"] = 42
    if "n_jobs" in p:
        kw["n_jobs"] = THREADS
    return kw


def suggest_params(trial, name):
    '''Optuna search space per CURATED model name (see 02_lazy.md L5 for the source ranges).'''
    if name == "LinearRegression":
        return {"fit_intercept": trial.suggest_categorical("fit_intercept", [True, False])}
    if name == "Ridge":
        return {"alpha": trial.suggest_float("alpha", 1e-3, 100, log=True)}
    if name == "Lasso":
        return {"alpha": trial.suggest_float("alpha", 1e-4, 10, log=True)}
    if name == "ElasticNet":
        return {"alpha": trial.suggest_float("alpha", 1e-4, 10, log=True),
                "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0)}
    if name == "Lars":
        return {"n_nonzero_coefs": trial.suggest_int("n_nonzero_coefs", 1, 50)}
    if name == "LassoLars":
        return {"alpha": trial.suggest_float("alpha", 1e-4, 10, log=True)}
    if name == "OrthogonalMatchingPursuit":
        return {"n_nonzero_coefs": trial.suggest_int("n_nonzero_coefs", 1, 50)}
    if name == "BayesianRidge":
        return {"alpha_1": trial.suggest_float("alpha_1", 1e-8, 1e-2, log=True),
                "alpha_2": trial.suggest_float("alpha_2", 1e-8, 1e-2, log=True),
                "lambda_1": trial.suggest_float("lambda_1", 1e-8, 1e-2, log=True),
                "lambda_2": trial.suggest_float("lambda_2", 1e-8, 1e-2, log=True)}
    if name == "HuberRegressor":
        return {"epsilon": trial.suggest_float("epsilon", 1.05, 5.0),
                "alpha": trial.suggest_float("alpha", 1e-6, 1.0, log=True)}
    if name == "SGDRegressor":
        return {"alpha": trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
                "penalty": trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"]),
                "learning_rate": trial.suggest_categorical("learning_rate", ["invscaling", "optimal", "adaptive"]),
                "eta0": trial.suggest_float("eta0", 1e-4, 1e-1, log=True)}
    if name == "PassiveAggressiveRegressor":
        return {"C": trial.suggest_float("C", 1e-3, 10, log=True),
                "epsilon": trial.suggest_float("epsilon", 1e-3, 1.0, log=True)}
    if name == "LinearSVR":
        return {"C": trial.suggest_float("C", 1e-3, 10, log=True),
                "epsilon": trial.suggest_float("epsilon", 1e-3, 1.0, log=True)}
    if name == "KNeighborsRegressor":
        return {"n_neighbors": trial.suggest_int("n_neighbors", 3, 100),
                "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
                "p": trial.suggest_categorical("p", [1, 2])}
    if name in ("DecisionTreeRegressor", "ExtraTreeRegressor"):
        depth_none = trial.suggest_categorical("max_depth_none", [True, False])
        return {"max_depth": None if depth_none else trial.suggest_int("max_depth", 3, 30),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
                "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.5, 0.8, None])}
    if name in ("RandomForestRegressor", "ExtraTreesRegressor"):
        depth_none = trial.suggest_categorical("max_depth_none", [True, False])
        sqrt_feat = trial.suggest_categorical("max_features_sqrt", [True, False])
        return {"n_estimators": trial.suggest_int("n_estimators", 100, 600),
                "max_depth": None if depth_none else trial.suggest_int("max_depth", 5, 40),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
                "max_features": "sqrt" if sqrt_feat else trial.suggest_float("max_features_frac", 0.3, 1.0)}
    if name == "BaggingRegressor":
        return {"n_estimators": trial.suggest_int("n_estimators", 10, 200),
                "max_samples": trial.suggest_float("max_samples", 0.3, 1.0),
                "max_features": trial.suggest_float("max_features", 0.3, 1.0)}
    if name == "GradientBoostingRegressor":
        return {"n_estimators": trial.suggest_int("n_estimators", 100, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "max_depth": trial.suggest_int("max_depth", 2, 8),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50)}
    if name == "HistGradientBoostingRegressor":
        # early_stopping fixed False (not tuned): its default holds out a random, non-grouped
        # 10% internally, which leaks across cv_group; max_iter is tuned directly instead.
        return {"learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "max_iter": trial.suggest_int("max_iter", 100, 1000),
                "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 255),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 200),
                "l2_regularization": trial.suggest_float("l2_regularization", 0.0, 10.0),
                "early_stopping": False}
    if name == "AdaBoostRegressor":
        return {"n_estimators": trial.suggest_int("n_estimators", 50, 500),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 2.0, log=True),
                "loss": trial.suggest_categorical("loss", ["linear", "square", "exponential"])}
    if name == "MLPRegressor":
        sizes = [(64,), (128, 64), (256, 128), (128, 128, 64)]
        idx = trial.suggest_categorical("hidden_layer_sizes_idx", list(range(len(sizes))))
        return {"hidden_layer_sizes": sizes[idx],
                "alpha": trial.suggest_float("alpha", 1e-6, 1e-2, log=True),
                "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)}
    if name == "LGBMRegressor":
        return {"n_estimators": trial.suggest_int("n_estimators", 100, 2000),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 15, 255),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10, log=True)}
    raise ValueError(f"no search space defined for {name}")


def build_estimator(name, params):
    '''CURATED model, fully configured: tuned params + fixed extras + random_state/n_jobs
    where supported, scaled inside a Pipeline for the model families that need it.'''
    cls = CURATED_MAP[name]
    kwargs = {**params, **FIXED_EXTRA.get(name, {}), **_fixed_kwargs(cls)}
    model = cls(**kwargs)
    if name in SCALED_MODELS:
        return Pipeline([("scaler", StandardScaler()), ("model", model)])
    return model

In [6]:
# tuning data: a larger (but still budget-capped) subsample of the FULL train set, with its
# cv groups intact for GroupKFold -- separate from the smaller screening subsample above.
X_tune, y_tune, g_tune = subsample(d.X_train, d.y_train, d.g_train, budget(30000, 3000), seed=42)
gkf = GroupKFold(n_splits=3)
print("tuning subsample:", len(X_tune), "rows, groups:", len(np.unique(g_tune)))


def make_objective(name):
    def objective(trial):
        params = suggest_params(trial, name)
        # stash the *resolved* sklearn kwargs (not trial.params, which has raw helper keys like
        # "max_depth_none"/"hidden_layer_sizes_idx" that suggest_params() consumes but isn't a
        # valid constructor kwarg itself) so L6 can refit with exactly what this trial scored.
        trial.set_user_attr("model_params", params)
        fold_mae = []
        for tr_idx, va_idx in gkf.split(X_tune, y_tune, groups=g_tune):
            model = build_estimator(name, params)
            model.fit(X_tune.iloc[tr_idx], y_tune.iloc[tr_idx])
            pred = model.predict(X_tune.iloc[va_idx])
            fold_mae.append(metrics(y_tune.iloc[va_idx], pred)["mae"])
        return float(np.mean(fold_mae))
    return objective


studies = {}
for name in top3:
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_objective(name), n_trials=budget(40, 3), timeout=budget(900, 30), n_jobs=1)
    studies[name] = study
    print(f"{name}: {len(study.trials)} trials, best CV MAE={study.best_value:.2f}s, params={study.best_params}")

tuning subsample: 30000 rows, groups: 5


C:\Users\danma\Documents\Dan\Projects\Stinger Proj\.venv-mc\Lib\site-packages\sklearn\utils\_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)
C:\Users\danma\Documents\Dan\Projects\Stinger Proj\.venv-mc\Lib\site-packages\sklearn\utils\_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)
C:\Users\danma\Documents\Dan\Projects\Stinger Proj\.venv-mc\Lib\site-packages\sklearn\utils\_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)


C:\Users\danma\Documents\Dan\Projects\Stinger Proj\.venv-mc\Lib\site-packages\sklearn\utils\_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)


OrthogonalMatchingPursuit: 40 trials, best CV MAE=239.02s, params={'n_nonzero_coefs': 4}


C:\Users\danma\Documents\Dan\Projects\Stinger Proj\.venv-mc\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.367657e+08, tolerance: 1.920e+05
  model = cd_fast.enet_coordinate_descent(


...

C:\Users\danma\Documents\Dan\Projects\Stinger Proj\.venv-mc\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.055597e+05, tolerance: 1.920e+05
  model = cd_fast.enet_coordinate_descent(


C:\Users\danma\Documents\Dan\Projects\Stinger Proj\.venv-mc\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.312196e+06, tolerance: 2.189e+05
  model = cd_fast.enet_coordinate_descent(


C:\Users\danma\Documents\Dan\Projects\Stinger Proj\.venv-mc\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.678148e+07, tolerance: 2.201e+05
  model = cd_fast.enet_coordinate_descent(


ElasticNet: 40 trials, best CV MAE=230.22s, params={'alpha': 9.165864194927282, 'l1_ratio': 0.6923717541676393}


GradientBoostingRegressor: 10 trials, best CV MAE=211.04s, params={'n_estimators': 650, 'learning_rate': 0.019721610970574007, 'max_depth': 5, 'subsample': 0.7962072844310213, 'min_samples_leaf': 3}


## L6 — Final fit + test
Refit each tuned model on the **full** train split (timed), predict test once, `save_result`.
`extra` records the screening rank/val R², the top-10 leaderboard, the final-refit wall time
(`fit_seconds`) and the number of Optuna trials actually completed (`n_trials_complete`, which
can be below `budget(40, 3)` when the per-model timeout hits first) for traceability.

In [7]:
leaderboard_top10 = ranked.head(10).reset_index().to_dict(orient="records")
results = {}
for rank, name in enumerate(top3, start=1):
    study = studies[name]
    best_params = study.best_trial.user_attrs["model_params"]
    final_model = build_estimator(name, best_params)
    t0 = time.time()
    final_model.fit(d.X_train, d.y_train)
    fit_seconds = time.time() - t0
    y_pred = final_model.predict(d.X_test)
    results[name] = save_result(
        DS, "lazy", name, d.y_test, y_pred,
        params=best_params,
        cv_mae=study.best_value,
        extra={"screen_rank": rank, "screen_val_r2": float(ranked.loc[name, "R-Squared"]),
               "leaderboard_top10": leaderboard_top10, "fit_seconds": fit_seconds,
               "n_trials_complete": len(study.trials)},
    )

[combined__lazy__OrthogonalMatchingPursuit] test R2=0.0894  MAE=191.3s  RMSE=265.9s  (n=29141)


[combined__lazy__ElasticNet] test R2=0.1358  MAE=200.7s  RMSE=259.0s  (n=29141)


[combined__lazy__GradientBoostingRegressor] test R2=0.2442  MAE=156.6s  RMSE=242.2s  (n=29141)


## L7 — Summary
Test metrics for the 3 tuned models, including each one's final-refit fit time.

In [8]:
summary = pd.DataFrame([
    {"dataset": DS, "family": r["family"], "model": r["model"], "screen_rank": r["extra"]["screen_rank"],
     "r2": r["r2"], "mae": r["mae"], "rmse": r["rmse"], "n_test": r["n_test"], "cv_mae": r["cv_mae"],
     "fit_seconds": r["extra"]["fit_seconds"], "n_trials_complete": r["extra"]["n_trials_complete"]}
    for r in results.values()
]).sort_values("r2", ascending=False).reset_index(drop=True)
summary

,dataset,family,model,screen_rank,r2,mae,rmse,n_test,cv_mae,fit_seconds,n_trials_complete
0,combined,lazy,GradientBoostingRegressor,3,0.244212,156.573352,242.232835,29141,211.039888,207.232935,10
1,combined,lazy,ElasticNet,2,0.135826,200.741367,259.020102,29141,230.219953,0.115276,40
2,combined,lazy,OrthogonalMatchingPursuit,1,0.089422,191.259621,265.883557,29141,239.015034,0.090613,40
